# Prepare test scenario

This Notebook modifies the Bentley base Strategy City scenario to serve as a simple test network for gtamodel_tools.

This Notebook must be run using Notebook included in Emme.

In [ ]:
from math import cos, pi, sin
import numpy as np
from pathlib import Path

# Emme imports
import inro.modeller as _m

# Define Modeller and desired tools
mm = _m.Modeller()
adjust_network_coordinates = mm.tool(
    "inro.emme.data.network.base.adjust_network_coordinates"
)
assign_sola = mm.tool(
    "inro.emme.traffic_assignment.sola_traffic_assignment")
assign_transit = mm.tool(
    "inro.emme.transit_assignment.extended_transit_assignment")
change_db_dim = mm.tool(
    "inro.emme.data.database.change_database_dimensions")
change_line_vehicle = mm.tool(
    "inro.emme.data.network.transit.change_line_vehicle")
create_function = mm.tool(
    "inro.emme.data.function.create_function")
create_mode = mm.tool(
    "inro.emme.data.network.mode.create_mode")
export_binary_matrix = mm.tool(
    "tmg.input_output.export_binary_matrix")
export_nwp = mm.tool(
    "tmg.input_output.export_network_package")

# Define emmebank and scenarios
eb = mm.emmebank
orig_scen_id = 1
base_scen_id = 13
amauto_scen_id = 14
amtransit_scen_id = 15
pmauto_scen_id = 16
pmtransit_scen_id = 17

# Prepare outputs directory
eb_fp = Path(eb.path)
root_dir = eb_fp.parents[1]
media_dir = root_dir / "Media"

output_root_dir = root_dir / "Media" / 'Test_model_outputs'
output_root_dir.mkdir(exist_ok=True)

networks_dir = output_root_dir / 'Networks'
networks_dir.mkdir(exist_ok=True)

demand_dir = output_root_dir / 'Demand'
demand_dir.mkdir(exist_ok=True)

los_dir = output_root_dir / 'LOS Matrices'
los_am_dir = los_dir / 'AM'
los_pm_dir = los_dir / 'PM'
los_dir.mkdir(exist_ok=True)
los_am_dir.mkdir(exist_ok=True)
los_pm_dir.mkdir(exist_ok=True)

### Modify Base Network

* Make a fresh copy of scenario 1
* Scale units by a factor of 100: 
* Rotate around centroid (previous line by 17°) to match Toronto orientation
* Move to Toronto region
* Change the database user coordinate to 0.001
*Prepare for auto assignment using the following steps
    * Create an auto mode that we can run traffic assignments
    * Add two functions:
        ○ fd1: (length*60/ul2)*(1+(volau/(lanes*ul3))^4)
        ○ fd2: (length*60/ul2)		
    * Create two auto demand matrix as follows:
        * mf2 - AM Auto Demand
|         | zonA(1) | zonX(3) | zonY(4) | zonB(5) |        			
| ------- | ------- | ------- | ------- | ------- |
| zonA(1) |     0	|  100    |     0   |    100  |
| zonX(3) |     0	|    0    |     0   |      0  |
| zonY(4) |     0	|    0    |     0   |      0  |
| zonB(5) |     0   |    0    |     0   |      0  |

        * mf3 - PM Auto Demand (Reverse direction)
|         | zonA(1) | zonX(3) | zonY(4) | zonB(5) |        			
| ------- | ------- | ------- | ------- | ------- |
| zonA(1) |     0	|    0    |     0   |      0  |
| zonX(3) |   100	|    0    |     0   |      0  |
| zonY(4) |     0	|    0    |     0   |      0  |
| zonB(5) |   100   |    0    |     0   |      0  |

        * mf4 - AM Transit Demand
|         | zonA(1) | zonX(3) | zonY(4) | zonB(5) |        			
| ------- | ------- | ------- | ------- | ------- |
| zonA(1) |     0	|    0    |     0   |    100  |
| zonX(3) |     0	|    0    |     0   |      0  |
| zonY(4) |     0	|    0    |     0   |      0  |
| zonB(5) |     0   |    0    |     0   |      0  |

        * mf5 - PM Transit Demand
|         | zonA(1) | zonX(3) | zonY(4) | zonB(5) |        			
| ------- | ------- | ------- | ------- | ------- |
| zonA(1) |     0	|    0    |     0   |      0  |
| zonX(3) |     0	|    0    |     0   |      0  |
| zonY(4) |     0	|    0    |     0   |      0  |
| zonB(5) |   100   |    0    |     0   |      0  |

        * mf10 - AM Auto Skim
        * mf11 - PM Auto Skim
        * mf12 - AM Transit
        * mf13 = AM Transit

In [ ]:
for scen_id in [base_scen_id, amauto_scen_id, amtransit_scen_id, pmauto_scen_id, pmtransit_scen_id]:
    if eb.scenario(scen_id) is not None:
        eb.delete_scenario(scen_id)

for mtx_id in ['mf2', 'mf3', 'mf4', 'mf5', 'mf10', 'mf11', 'mf12', 'mf13']:
    if eb.matrix(mtx_id) is not None:
        eb.delete_matrix(mtx_id)
 
base_scen = eb.copy_scenario(
    orig_scen_id, 
    base_scen_id, 
    copy_path_files=False, 
    copy_strat_files=False
)
base_scen.has_traffic_results = False
base_scen.has_transit_results = False

# Use the Adjust Network Coordinates tool to scale, move and 
# rotate to match Toronto scale
a = -17.0 * pi / 180.0
matrix_scale = '0,0,100,100,0,0,0,0'
matrix_rotate = f'0,0,{cos(a)},{cos(a)},0,0,{sin(a)},{-sin(a)}'
matrix_translate = '-626500,-4832300,1,1,0,0,0,0'
adjust_network_coordinates(
    scenario=base_scen,  
    transformation_matrix=matrix_scale, 
) 
adjust_network_coordinates(
    scenario=base_scen,  
    transformation_matrix=matrix_rotate, 
) 
adjust_network_coordinates(
    scenario=base_scen,  
    transformation_matrix=matrix_translate, 
) 

eb.coord_unit_length = 0.001

auto_mode = create_mode(
    mode_type='AUTO', 
    mode_id='c',
    mode_description='auto',
    scenario=base_scen
)

for f in ['fd1', 'fd2']:
    if eb.function(f) is not None:
        eb.delete_function(f)
fd1 = create_function(
    function_id='fd1',
    function_expression='(length*60/ul2)*(1+(volau/(lanes*ul3))^4)'
)
fd2 = create_function(
    function_id='fd2', 
    function_expression='length*60/ul2'
)

# Create demand matrices
mtx = eb.create_matrix('mf2')
mtx.name = 'demAutoAM'
mtx.description = 'AM Auto Demand'
mtx.set_numpy_data(
    np.array([
            [  0, 100, 0, 100], 
            [  0,   0, 0,   0],
            [  0,   0, 0,   0],
            [  0,   0, 0,   0]
        ]
    )
)

mtx = eb.create_matrix('mf3')
mtx.name = 'demAutoPM'
mtx.description = 'PM Auto Demand'
mtx.set_numpy_data(
    np.array([
            [  0,   0, 0,   0], 
            [100,   0, 0,   0],
            [  0,   0, 0,   0],
            [100,   0, 0,   0]
        ]
    )
)

mtx = eb.create_matrix('mf4')
mtx.name = 'demTransitAM'
mtx.description = 'AM Transit Demand'
mtx.set_numpy_data(
    np.array([
            [  0,   0, 0, 100], 
            [  0,   0, 0,   0],
            [  0,   0, 0,   0],
            [  0,   0, 0,   0]
        ]
    )
)

mtx = eb.create_matrix('mf5')
mtx.name = 'demTransitPM'
mtx.description = 'PM Transit Demand'
mtx.set_numpy_data(
    np.array([
            [  0,   0, 0,   0], 
            [100,   0, 0,   0],
            [  0,   0, 0,   0],
            [100,   0, 0,   0]
        ]
    )
)

mtx = eb.create_matrix('mf10')
mtx.name = 'AM_aivtt'
mtx.description = 'AM Auto IVTT'
mtx = eb.create_matrix('mf11')
mtx.name = 'PM_aivtt'
mtx.description = 'PM Auto IVTT'

This next cell uses the Network API to make the following changes:

* Split links 101-105 and 105-101 at their vertices. This will allow more options when aggregating volumes, etc.
* Set the length of all regular links to the shape length
* Set the length of all connector links to 200 m
* Setup network so that we can run an auto assignment
    * for connectors, set vdf: 2, ul2: 30, ul3: 9999
    * for regular links, set vdf: 1, ul2: 40, ul3:500
* Create a reverse of each transit line

In [ ]:
# Helper functions
def is_connector(l):
    if l.i_node.is_centroid or l.j_node.is_centroid:
        return True
    else:
        return False

def create_node(net, node_id, x, y):
    if net.node(node_id) is None:
        n = net.create_regular_node(node_id)
    else:
        n = net.node(node_id)
    n.x = x
    n.y = y
    return n 

def create_sublink(net, from_link, inode_id, jnode_id):
    modes = from_link.modes
    l = net.create_link(inode_id, jnode_id, modes)
    l.length = l.shape_length
    l.type = from_link.type
    l.num_lanes = 2   # changing lanes to differentiate lane-km tests
    l.volume_delay_func = from_link.volume_delay_func
    l.data1 = from_link.data1
    l.data2 = from_link.data2
    l.data3 = from_link.data3    
    
def copy_transit_line(net, line, itinerary, name):
    veh_id = line.vehicle.number
    nl = net.create_transit_line(name, veh_id, itinerary)
    
    
    nl.description = line.description
    nl.headway = line.headway
    nl.speed = line.speed
    nl.layover_time = line.layover_time
    nl.data1 = line.data1
    nl.data2 = line.data2
    nl.data3 = line.data3
    return nl

In [ ]:
# Network changes
net = base_scen.get_network()

# Create a new lrt mode and vehicle
mode_l = net.create_mode('TRANSIT', 'l')
mode_l.description = 'lrt' 
veh_l = net.create_transit_vehicle(2, 'l')
veh_l.description = 'lrt'
veh_l.auto_equivalent = 3.0
veh_l.seated_capacity = 40
veh_l.total_capacity = 100

# Add mode l to links 104-105 and 105-104
net.link(104, 105).modes |= set([net.mode('l')])
net.link(105, 104).modes |= set([net.mode('l')])

# Split links 101-105 and 105-101 at all vertices
l_101_105 = net.link(101, 105)
l_105_101 = net.link(105, 101)
first_node = 101
last_node = 105
inode_id = first_node
jnode_id = 200
itinerary = [inode_id]
for v in l_101_105.vertices:
    create_node(net, jnode_id, v[0], v[1])
    create_sublink(net, l_101_105, inode_id, jnode_id)
    create_sublink(net, l_105_101, jnode_id, inode_id)
    itinerary.append(jnode_id)
    inode_id = jnode_id
    jnode_id += 1
jnode_id = last_node
create_sublink(net, l_101_105, inode_id, jnode_id)
create_sublink(net, l_105_101, jnode_id, inode_id)
itinerary.append(jnode_id)

# Before we can delete the original links, we need to modify the
# Red line to use the new (split) links. Note that we cannot change 
# the itinerary of an existing line, hence we need to create a new one
# and delete the original one afterwards.
red_line = net.transit_line("Red")
temp_line = copy_transit_line(net, red_line, itinerary, "Red2")
net.delete_transit_line("Red")
new_line = net.copy_transit_line("Red2", "Red")
net.delete_transit_line("Red2")

# Now we can delete the original links
net.delete_link(101, 105)
net.delete_link(105, 101)

# Set the link length to the shape length for regular links
# or to 200m for connectors
for l in net.links():
    if is_connector(l):
        l.length = 0.2
    else:
        l.length = l.shape_length

# Add auto mode and set vdf, ul2 and ul3 on all links
for l in net.links():
    l.modes |= set([auto_mode])
    if is_connector(l):
        l.volume_delay_func = 2
        l.data2 = 30
        l.data3 = 9999
    else:
        l.volume_delay_func = 1
        l.data2 = 40
        l.data3 = 500
        

    
# Create a reverse-direction copy of all transit lines
for tl in net.transit_lines():
    new_line_id = f'{tl.id}_rev'
    net.copy_transit_line(tl.id, new_line_id, reverse_itinerary=True)
        
base_scen.publish_network(net)

In [ ]:
# Change Black line to LRT mode
change_line_vehicle(vehicle=2,selection="Black____", scenario=base_scen)

### Run SOLA auto assignment

The completed auto assignment has the following:
* From A->B, 590.8257 people passing on the bottom route, and 409.1743 people passing on the top route. This is because demand is higher than the capacity for the bottomm route alone.
* from B->A, the demand is less than that of a single route, hence everyone takes the bottom route.

In [ ]:
traffic_assignment_spec = {
    "type": "SOLA_TRAFFIC_ASSIGNMENT",
    "classes": [
        {
            "mode": "c",
            "demand": None,
            "generalized_cost": None,
            "results": {
                "link_volumes": None,
                "turn_volumes": None,
                "od_travel_times": {
                    "shortest_paths": None
                }
            },
            "path_analyses": []
        }
    ],
    "performance_settings": {
        "number_of_processors": "max",
        "network_acceleration": False,
        "u_turns_allowed": False
    },
    "background_traffic": None,
    "stopping_criteria": {
        "max_iterations": 100,
        "relative_gap": 0.0001,
        "best_relative_gap": 0.01,
        "normalized_gap": 0.005
    }
}

transit_assignment_spec = {
    "type": "EXTENDED_TRANSIT_ASSIGNMENT",
    "modes": [
        "b",
        "a",
        "w",
        "l"
    ],
    "demand": None,
    "waiting_time": {
        "headway_fraction": 0.5,
        "effective_headways": "hdw",
        "spread_factor": 1,
        "perception_factor": 1
    },
    "boarding_time": {
        "global": {
            "penalty": 0,
            "perception_factor": 1
        },
        "at_nodes": None,
        "on_lines": None,
        "on_segments": None
    },
    "boarding_cost": {
        "global": {
            "penalty": 0,
            "perception_factor": 1
        },
        "at_nodes": None,
        "on_lines": None,
        "on_segments": None
    },
    "in_vehicle_time": {
        "perception_factor": 1
    },
    "in_vehicle_cost": None,
    "aux_transit_time": {
        "perception_factor": 1
    },
    "aux_transit_cost": None,
    "aux_transit_by_mode": None,
    "flow_distribution_at_origins": {
        "choices_at_origins": "OPTIMAL_STRATEGY",
        "fixed_proportions_on_connectors": None
    },
    "flow_distribution_at_regular_nodes_with_aux_transit_choices": {
        "choices_at_regular_nodes": "OPTIMAL_STRATEGY"
    },
    "flow_distribution_between_lines": {
        "consider_total_impedance": False
    },
    "connector_to_connector_path_prohibition": None,
    "circular_lines": {
        "stay": False
    },
    "od_results": {
        "total_impedance": None
    },
    "results": {
        "aux_transit_volumes_by_mode": [
            {
                "mode": "a",
                "volume": None
            },
            {
                "mode": "w",
                "volume": None
            }
        ]
    },
    "journey_levels": [],
    "performance_settings": {
        "number_of_processors": "max"
    }
}

In [ ]:
# AM Auto and transit assignments
amauto_scen = eb.copy_scenario(base_scen_id, amauto_scen_id)
traffic_assignment_spec['classes'][0]['demand'] = 'mf2'
traffic_assignment_spec['classes'][0]['results']['od_travel_times']['shortest_paths'] = 'mf10'
assign_sola(traffic_assignment_spec, scenario=amauto_scen)

amtransit_scen = eb.copy_scenario(base_scen_id, amtransit_scen_id)
transit_assignment_spec['demand'] = 'mf4'
assign_transit(transit_assignment_spec, save_strategies=False, scenario=amtransit_scen)

In [ ]:
# PM Auto and transit assignments
pmauto_scen = eb.copy_scenario(base_scen_id, pmauto_scen_id)
traffic_assignment_spec['classes'][0]['demand'] = 'mf3'
traffic_assignment_spec['classes'][0]['results']['od_travel_times']['shortest_paths'] = 'mf11'
assign_sola(traffic_assignment_spec, scenario=pmauto_scen)

pmtransit_scen = eb.copy_scenario(base_scen_id, pmtransit_scen_id)
transit_assignment_spec['demand'] = 'mf5'
assign_transit(transit_assignment_spec, save_strategies=False, scenario=pmtransit_scen)

In [ ]:
# Export networks
export_nwp(amauto_scen_id, (networks_dir / 'AM_Auto.nwp').as_posix(), 'all')
export_nwp(amtransit_scen_id, (networks_dir / 'AM_Transit.nwp').as_posix(), 'all')
export_nwp(pmauto_scen_id, (networks_dir / 'PM_Auto.nwp').as_posix(), 'all')
export_nwp(pmtransit_scen_id, (networks_dir / 'PM_Transit.nwp').as_posix(), 'all')

In [ ]:
# Export Matrices
xtmf_fullmat_type = 4

# Auto Demand Matrices
fp = (demand_dir / 'AM_Auto.mtx').as_posix()
export_binary_matrix(
    xtmf_fullmat_type, 2, fp, amauto_scen_id
)
fp = (demand_dir / 'PM_Auto.mtx').as_posix()
export_binary_matrix(
    xtmf_fullmat_type, 3, fp, pmauto_scen_id
)

# Transit Demand Matrices
fp = (demand_dir / 'AM_Transit.mtx').as_posix()
export_binary_matrix(
    xtmf_fullmat_type, 4, fp, amtransit_scen_id
)
fp = (demand_dir / 'PM_Tansit.mtx').as_posix()
export_binary_matrix(
    xtmf_fullmat_type, 5, fp, pmtransit_scen_id
)

# Auto Skim matrices
fp = (los_am_dir / 'aivtt.mtx').as_posix()
export_binary_matrix(
    xtmf_fullmat_type, 10, fp, amauto_scen_id
)
fp = (los_pm_dir / 'aivtt.mtx').as_posix()
export_binary_matrix(
    xtmf_fullmat_type, 11, fp, pmauto_scen_id
)
